# Análise Exploratória: Séries Temporais de Dengue

Este notebook realiza a análise exploratória dos dados de dengue coletados via API do InfoDengue para 8 capitais brasileiras no período de 2010 a 2024.

In [ ]:
import glob
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', palette='colorblind')

CITY_LABELS = {
    'sao': 'São Paulo',
    'rio': 'Rio de Janeiro',
    'belo': 'Belo Horizonte',
    'brasilia': 'Brasília',
    'fortaleza': 'Fortaleza',
    'recife': 'Recife',
    'manaus': 'Manaus',
    'salvador': 'Salvador',
}

## 1. Carregamento dos dados

In [ ]:
series_dict = {}
for fpath in sorted(glob.glob('../data/processed/dengue_monthly_*.csv')):
    city_key = Path(fpath).stem.replace('dengue_monthly_', '')
    series = pd.read_csv(fpath, index_col='date', parse_dates=['date'])['value']
    series_dict[CITY_LABELS.get(city_key, city_key)] = series

print(f'Cidades carregadas: {list(series_dict.keys())}')

## 2. Estatísticas descritivas

In [ ]:
stats = pd.DataFrame({
    city: {
        'n_meses': len(s),
        'total_casos': int(s.sum()),
        'media_mensal': round(s.mean(), 0),
        'mediana_mensal': round(s.median(), 0),
        'max_mensal': int(s.max()),
        'data_pico': str(s.idxmax().date()),
    }
    for city, s in series_dict.items()
}).T

stats

## 3. Visualização das séries temporais

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for i, (city_name, series) in enumerate(series_dict.items()):
    axes[i].plot(series.index, series.values, color='steelblue', linewidth=1)
    axes[i].set_title(city_name, fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Ano')
    axes[i].set_ylabel('Casos mensais')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Séries Temporais de Casos de Dengue — Capitais Brasileiras (2010–2024)', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Sazonalidade mensal (boxplot)

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

month_names = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

for i, (city_name, series) in enumerate(series_dict.items()):
    df_month = pd.DataFrame({'mes': series.index.month, 'casos': series.values})
    sns.boxplot(data=df_month, x='mes', y='casos', ax=axes[i], color='steelblue')
    axes[i].set_title(city_name, fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Mês')
    axes[i].set_ylabel('Casos mensais')
    axes[i].set_xticklabels(month_names)

plt.suptitle('Sazonalidade Mensal de Dengue — Capitais Brasileiras', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Correlação entre cidades

In [ ]:
df_all = pd.DataFrame(series_dict)
corr = df_all.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlação de Pearson entre Séries de Dengue', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()